This program is for making bedfiles for the alternative differential binding calling programs used in the manuscript (e.g. the adhoc method, diff-skipper)

# Load necessary packages

In [1]:
import os
import numpy as np
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3

# Helper functions

In [2]:
def make_diff_skip_bed(path_up, path_down, output_file):
    # Load the "up" direction diff-skipper run. 
    diff_skip_up = pd.read_csv(path_up, sep = "\t")
    diff_skip_up["category"] = "up"

    # Load the "down" direction diff-skipper run. 
    diff_skip_down = pd.read_csv(path_down,sep = "\t")
    diff_skip_down["category"] = "down"

    # Concatonate together.
    diff_skip = pd.concat([diff_skip_up, diff_skip_down])

    # Setup config names.
    track_name = "DiffSkip Differential Windows"
    track_desc = "Up=red, Down=blue"
    output_dir = "beds" 
    
    # Ensure everything is the proper format. 
    diff_skip["start"] = pd.to_numeric(diff_skip["start"], errors="coerce").astype("Int64")
    diff_skip["end"]   = pd.to_numeric(diff_skip["end"], errors="coerce").astype("Int64")
    diff_skip["category"] = diff_skip["category"].astype(str).str.lower()
    
    # 1000 for up/down, else 0.
    diff_skip["score"] = np.where(diff_skip["category"].isin(["up", "down"]), 1000, 0).astype(int)
    
    # Assign colors. 
    diff_skip["itemRgb"] = "128,128,128"   # default gray
    diff_skip.loc[diff_skip["category"] == "up", "itemRgb"] = "255,0,0"
    diff_skip.loc[diff_skip["category"] == "down", "itemRgb"] = "0,0,255"

    # Create arbitrary name column. 
    diff_skip["name"] = np.arange(1, len(diff_skip) + 1).astype(str)
    
    # Create a strand placeholder, change start and end to thickstart and thickend. 
    diff_skip["strand"] = "."
    diff_skip["thickStart"] = diff_skip["start"]
    diff_skip["thickEnd"] = diff_skip["end"]
    
    # Assembl complete bedfile.
    bed = diff_skip[["chr", "start", "end", "name", "score", "strand",
                     "thickStart", "thickEnd", "itemRgb"]]
    
    # Write bedfile. 
    out_bed = os.path.join(output_file)
    os.makedirs(os.path.dirname(out_bed), exist_ok=True)
    
    with open(out_bed, "w") as fh:
        fh.write(
            f'track name="{track_name}" '
            f'description="{track_desc}" '
            f'itemRgb="On" visibility=3\n'
        )
        bed.to_csv(fh, sep="\t", header=False, index=False)

In [3]:
def make_adhoc_bed(path_trt, path_ctrl, output_file):

    # Load the treatment and control results. 
    adhoc_trt = pd.read_csv(path_trt, sep="\t")
    adhoc_ctrl = pd.read_csv(path_ctrl, sep="\t")

    # Define key columns for merging. 
    key_cols = ["chr", "start", "end"]

    # Perform merger. 
    adhoc_up = (adhoc_trt.merge(adhoc_ctrl[key_cols], on=key_cols, how="left", indicator=True).query('_merge == "left_only"').drop(columns="_merge"))
    adhoc_up["category"] = "up"
    adhoc_down = (adhoc_ctrl.merge(adhoc_trt[key_cols], on=key_cols, how="left", indicator=True).query('_merge == "left_only"').drop(columns="_merge"))
    adhoc_down["category"] = "down"

    # Concatenate. 
    adhoc = pd.concat([adhoc_up,adhoc_down])

    # Setup config names.
    track_name = "Adhoc Differential Windows"
    track_desc = "Up=red, Down=blue"
    output_dir = "beds"
    
    # Ensure everything is the proper format. 
    adhoc["start"] = pd.to_numeric(adhoc["start"], errors="coerce").astype("Int64")
    adhoc["end"]   = pd.to_numeric(adhoc["end"], errors="coerce").astype("Int64")
    adhoc["category"] = adhoc["category"].astype(str).str.lower()
    
    # 1000 for up/down, else 0.
    adhoc["score"] = np.where(adhoc["category"].isin(["up", "down"]), 1000, 0).astype(int)
    
    # Assign colors. 
    adhoc["itemRgb"] = "128,128,128"   # default gray
    adhoc.loc[adhoc["category"] == "up", "itemRgb"] = "255,0,0"
    adhoc.loc[adhoc["category"] == "down", "itemRgb"] = "0,0,255"

    # Create arbitrary name column. 
    adhoc["name"] = np.arange(1, len(adhoc) + 1).astype(str)
    
    # Create a strand placeholder, change start and end to thickstart and thickend. 
    adhoc["strand"] = "."
    adhoc["thickStart"] = adhoc["start"]
    adhoc["thickEnd"] = adhoc["end"]
    
    # Assembl complete bedfile.
    bed = adhoc[["chr", "start", "end", "name", "score", "strand",
                     "thickStart", "thickEnd", "itemRgb"]]
    
    # Write bedfile. 
    out_bed = os.path.join(output_file)
    os.makedirs(os.path.dirname(out_bed), exist_ok=True)
    
    with open(out_bed, "w") as fh:
        fh.write(
            f'track name="{track_name}" '
            f'description="{track_desc}" '
            f'itemRgb="On" visibility=3\n'
        )
        bed.to_csv(fh, sep="\t", header=False, index=False)

# Make beds

## NONO

In [4]:
make_diff_skip_bed("~/scratch/new_skipper_test/testing/results/NONO_skipper_diff2/output/reproducible_enriched_windows/10R_DMSO.reproducible_enriched_windows.tsv.gz",
                  "~/scratch/new_skipper_test/testing/results/NONO_skipper_diff2/output/reproducible_enriched_windows/DMSO_10R.reproducible_enriched_windows.tsv.gz",
                  "beds/NONO_diffskip.bed")


/scratch/kflanagan/job_10836626/ipykernel_1078552/2570045329.py:11: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  diff_skip = pd.concat([diff_skip_up, diff_skip_down])


In [5]:
make_adhoc_bed("/tscc/lustre/ddn/scratch/kflanagan/new_skipper_test/testing/results/NONO_skipper_updated/output/reproducible_enriched_windows/10R.reproducible_enriched_windows.tsv.gz",
              "/tscc/lustre/ddn/scratch/kflanagan/new_skipper_test/testing/results/NONO_skipper_updated/output/reproducible_enriched_windows/DMSO.reproducible_enriched_windows.tsv.gz",
              "beds/NONO_adhoc.bed")


## DDX42

In [6]:
make_diff_skip_bed("~/scratch/new_skipper_test/testing/results/DDX42_skipper_diff/output/reproducible_enriched_windows/active_DMSO.reproducible_enriched_windows.tsv.gz",
                  "~/scratch/new_skipper_test/testing/results/DDX42_skipper_diff/output/reproducible_enriched_windows/DMSO_active.reproducible_enriched_windows.tsv.gz",
                  "beds/DDX42_diffskip.bed")


In [7]:
make_adhoc_bed("/tscc/lustre/ddn/scratch/kflanagan/new_skipper_test/testing/results/DDX42_skipper_updated/output/reproducible_enriched_windows/MJ-22-51_2A.reproducible_enriched_windows.tsv.gz",
              "/tscc/lustre/ddn/scratch/kflanagan/new_skipper_test/testing/results/DDX42_skipper_updated/output/reproducible_enriched_windows/MJ-22-51_1A.reproducible_enriched_windows.tsv.gz",
              "beds/DDX42_adhoc.bed")


## PUF60

In [8]:
make_diff_skip_bed("~/scratch/new_skipper_test/testing/results/PUF60_skipper_diff/output/reproducible_enriched_windows/L140P_WT.reproducible_enriched_windows.tsv.gz",
                  "~/scratch/new_skipper_test/testing/results/PUF60_skipper_diff/output/reproducible_enriched_windows/WT_L140P.reproducible_enriched_windows.tsv.gz",
                  "beds/PUF60_diffskip.bed")


In [9]:
make_adhoc_bed("/tscc/lustre/ddn/scratch/kflanagan/new_skipper_test/testing/results/PUF60_skipper_updated/output/reproducible_enriched_windows/L140P-PUF60.reproducible_enriched_windows.tsv.gz",
              "/tscc/lustre/ddn/scratch/kflanagan/new_skipper_test/testing/results/PUF60_skipper_updated/output/reproducible_enriched_windows/WT-PUF60.reproducible_enriched_windows.tsv.gz",
              "beds/PUF60_adhoc.bed")
